In [1]:
import pandas as pd

# Load the dataset
data = pd.read_pickle('../../data/processed/cleaned_articles.pkl')

In [2]:
data.info()
data.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39863 entries, 0 to 39862
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   title              39863 non-null  object
 1   publisher          39863 non-null  object
 2   date               39863 non-null  object
 3   section            39863 non-null  object
 4   body               39863 non-null  object
 5   length             39863 non-null  int64 
 6   letters_flag       39863 non-null  bool  
 7   country_flag       39863 non-null  object
 8   foreign_countries  39863 non-null  object
 9   us_mentions        39863 non-null  object
 10  source_file        39863 non-null  object
 11  publisher_raw      39863 non-null  object
 12  section_clean      39845 non-null  object
 13  section_category   34176 non-null  object
dtypes: bool(1), int64(1), object(12)
memory usage: 4.0+ MB


,title,publisher,date,section,body,length,letters_flag,country_flag,foreign_countries,us_mentions,source_file,publisher_raw,section_clean,section_category
0,A Well-Documented Childhood. A Very Private Life.,New York Times,2024-12-31,Section A; Column 0; National Desk; Pg. 16,Jimmy Carter's daughter had an extraordinary a...,1209,False,BOTH,"[Canada, Egypt, Mexico, Nicaragua]","[Massachusetts, Rhode Island, US, Washington]",NYT/1.DOCX,The New York Times,section a; column 0; national desk; pg. 16,US/National
1,These were the big stories in arts and culture...,Other publisher,2024-12-31,WHAT TO KNOW,The arts in Dayton continued to thrive in 2024...,3345,False,BOTH,"[France, Japan, Paris (Capital), Sierra Leone,...","[America, California, Florida, Georgia, Massac...",Other publishers/Files (500) (1).DOCX,Dayton Daily News (Ohio),what to know,None
2,She Exalted The Beauty Of Dance,New York Times,2024-12-31,Section C; Column 0; The Arts/Cultural Desk; P...,She was The New Yorker's first dance critic. H...,1314,False,BOTH,[Male (Capital)],"[New York, North Carolina, US, United States]",NYT/1.DOCX,The New York Times,section c; column 0; the arts/cultural desk; p...,Arts/Culture
3,"Under a Highway in Rio, a Dance Style Charms a...",New York Times,2024-12-31,WORLD; americas,"Trucks, buses and cars rumbled overhead, drown...",1333,False,BOTH,"[Brazil, Lima (Capital)]","[New York, US, United States]",NYT/1.DOCX,The New York Times,world; americas,World/International
4,"Congressional pay, minimum wage stagnant for y...",Other publisher,2024-12-31,OPINION; Pg. A13,ABSTRACT\nMembers of Congress have not seen a ...,921,False,US_ONLY,[],"[America, Delaware, Maryland, New Jersey, New ...",Other publishers/Files (500) (1).DOCX,The Philadelphia Inquirer,opinion; pg. a13,Opinion/Editorial/Letters


In [3]:
print(data.columns)


Index(['title', 'publisher', 'date', 'section', 'body', 'length',
       'letters_flag', 'country_flag', 'foreign_countries', 'us_mentions',
       'source_file', 'publisher_raw', 'section_clean', 'section_category'],
      dtype='object')


### Preprocess with custom stopwords

In [10]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ---- 1. Define custom stopwords ----
custom_stopwords = {
    'said', 'say', 'saying', 'told',
    'mr', 'ms', 'mrs',
    'report', 'reported', 'interview', 'according',
    'article', 'photo', 'photograph', 'graphic',
    'someone', 'anyone', 'everyone', 'everybody',
    'york', 'new', 'time','much', 'many', 'way', 'often', 'even', 'always', 'got', 'get', 'never', 'among','could',
    'would','last','also','still','whether','opinion','times','photos','photographs','number','year','month','day',
    'week','decade','know','thing','went','far','recent','lately', 'th','around','page','near','section facebook',
    'follow section','twitter nytopinion','nytopinion','facebook twitter','back','like','make','made','see','author'

}

# ---- 2. Merge with NLTK stopwords ----
stop_words = set(stopwords.words('english')).union(custom_stopwords)

# ---- 3. Lemmatizer ----
lemmatizer = WordNetLemmatizer()

# ---- 4. Preprocess function ----
def preprocess(text):
    text = text.lower()  
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep letters, spaces
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return ' '.join(words)

# ---- 5. Apply cleaning ----
data['clean_text'] = data['body'].astype(str).apply(preprocess)

data[['body', 'clean_text']].head()


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jingguo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/jingguo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,body,clean_text
0,Jimmy Carter's daughter had an extraordinary a...,jimmy carter daughter extraordinary well docum...
1,The arts in Dayton continued to thrive in 2024...,art dayton continued thrive exciting debut art...
2,She was The New Yorker's first dance critic. H...,yorker first dance critic wit devastating behi...
3,"Trucks, buses and cars rumbled overhead, drown...",truck bus car rumbled overhead drowning marcus...
4,ABSTRACT\nMembers of Congress have not seen a ...,abstract member congress seen pay raise since ...


### Vectorize (CountVectorizer)

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_features=50000,     # Limit vocabulary size for efficiency
    ngram_range=(1, 2)      # include unigrams + bigramslike "small business" and "self employment"

)

# Vectorize the clean_text column
X = vectorizer.fit_transform(data['clean_text'])

# Extract vocabulary
vocab = vectorizer.get_feature_names_out()

print("Vocabulary size:", len(vocab))
vocab[:30]   # Show the first 30 vocabulary items


Vocabulary size: 50000


array(['aa', 'aaa', 'aaron', 'aarp', 'aback', 'abaire', 'abandon',
       'abandoned', 'abandoned building', 'abandoned home',
       'abandoned house', 'abandoning', 'abandonment', 'abatement',
       'abbas', 'abbey', 'abbott', 'abbreviated', 'abby', 'abc',
       'abc news', 'abdel', 'abducted', 'abduction', 'abdul', 'abdullah',
       'abe', 'abel', 'aberration', 'abetted'], dtype=object)

### Define anchors + build anchor_indices + missing_anchors

In [41]:
# Step 3: Define anchor words 
anchor_words_dict = {
    'Housing': [
        'housing', 'rent', 'tenant', 'affordable', 'homelessness',
        'landlord', 'eviction', 'apartment', 'mortgage'
    ],
    'Education': [
        'education', 'school', 'college', 'teacher', 'university', 'tuition',
        'student', 'curriculum', 'classroom', 'learning'
    ],
    'Union': [
        'union', 'labor', 'strike', 'worker', 'collective',
        'organizing', 'wage', 'bargaining', 'employment'
    ],
    'Election': [
        'election', 'vote', 'campaign', 'party', 'candidate', 'rally',
        'politics', 'ballot', 'primary', 'poll', 'democrat', 'republican'
    ],
    'Military': [
        'military', 'service', 'veteran', 'army', 'soldier',
        'defense', 'troop', 'combat', 'deployment', 'navy','ptsd','active duty','civilian life','war'
    ],
    'Indebtedness': [
        'debt', 'Indebtedness','struggle', 'credit', 'bankrupt', 'bankruptcy',
        'loan', 'foreclosure', 'repayment', 'interest', 'borrower', 'default', 'financial'
    ],
    'Family': [
        'family', 'child', 'parent', 'marriage', 'spouse', 'household',
        'domestic', 'raising', 'sibling', 'divorce', 'wedding', 'extended family', 'kin'  # bigram
    ],
    'Art': [
        'art', 'review', 'theater', 'play', 'movie',
        'culture', 'film', 'gallery', 'music', 'performance', 'exhibition', 'artist'
    ],
    'Health': [
        'overdose', 'health','patient', 'disease', 'treatment', 'mental', 'therapy',
        'nurse', 'clinic', 'pandemic', 'loneliness', 'medicine', 'doctor','addiction','insurance','drug'
    ],
    'Business Ownership': [
        'business', 'entrepreneur', 'self employment','business owner','self employed',   
        'client', 'customer', 'firm', 'owner', 'small business'   # bigram
    ],
    'Sports': [
        'soccer', 'hockey', 'athlete', 'league', 'tournament', 'game', 'score','sport'
    ],
    'Religion': [
        'faith', 'islamic', 'sermon', 'worship', 'belief','religion', 'church', 'temple', 'mosque', 'christianity', 'catholic',
        'evangelical'
    ],
    'Crime': [
        'police', 'policeman', 'crime', 'theft', 'criminal','gunshot', 'homicide'
    ],
     'Books': [
        'book','author','novel','book review', 'write', 'wrote','poem','poetry','fiction','non fiction'
    ]
    
}

topic_names = list(anchor_words_dict.keys())

# Map words to vocabulary indices
word_to_idx = {w: i for i, w in enumerate(vocab)}

anchor_indices = []
for topic, words in anchor_words_dict.items():
    idx_list = [word_to_idx[w] for w in words if w in word_to_idx]
    anchor_indices.append(idx_list)
    print(f"{topic}: {len(idx_list)} anchors found out of {len(words)} total")


Housing: 9 anchors found out of 9 total
Education: 10 anchors found out of 10 total
Union: 9 anchors found out of 9 total
Election: 12 anchors found out of 12 total
Military: 13 anchors found out of 14 total
Indebtedness: 12 anchors found out of 13 total
Family: 13 anchors found out of 13 total
Art: 12 anchors found out of 12 total
Health: 16 anchors found out of 16 total
Business Ownership: 9 anchors found out of 10 total
Sports: 8 anchors found out of 8 total
Religion: 12 anchors found out of 12 total
Crime: 7 anchors found out of 7 total
Books: 9 anchors found out of 10 total


In [42]:
# Inspect which anchor words are missing from the vocabulary
missing_anchors = {}

for topic, words in anchor_words_dict.items():
    missing = [w for w in words if w not in word_to_idx]
    if missing:
        missing_anchors[topic] = missing

missing_anchors


{'Military': ['civilian life'],
 'Indebtedness': ['Indebtedness'],
 'Business Ownership': ['self-employment'],
 'Books': ['non-fiction']}

### Train corex2 (strength=5) + print topic

In [43]:
from corextopic import corextopic as ct

corex5 = ct.Corex(
    n_hidden=len(topic_names),
    seed=42
)

corex5.fit(
    X,
    words=vocab,
    anchors=anchor_indices,
    anchor_strength=5
)

print("Total correlation (TC):", corex5.tc)


Total correlation (TC): 152.37383153415465


In [44]:
n_words = 15  

for i, topic in enumerate(corex5.get_topics(n_words=n_words)):
    print(f"\n### Topic {i+1}: {topic_names[i]}")
    for word, score, idx in topic:
        print(f"{word:20} {score:.3f}")



### Topic 1: Housing
housing              0.667
apartment            0.573
rent                 0.418
tenant               0.305
affordable           0.223
landlord             0.202
city                 0.129
building             0.127
neighborhood         0.126
resident             0.117
street               0.083
area                 0.080
mortgage             0.080
park                 0.077
property             0.074

### Topic 2: Education
school               1.367
student              1.051
college              0.742
education            0.649
university           0.602
teacher              0.496
classroom            0.232
tuition              0.199
learning             0.149
curriculum           0.130
high school          0.128
graduate             0.059
high                 0.058
public school        0.052
grade                0.051

### Topic 3: Union
wage                 1.221
labor                0.920
worker               0.791
union                0.692
employment      

In [45]:
import pandas as pd

# ---------------------------------------------------
# 1. Put topic names here
# ---------------------------------------------------
topic_names = [
    "Housing",
    "Education",
    "Union",
    "Election",
    "Military",
    "Indebtedness",
    "Family",
    "Art",
    "Health",
    "Business Ownership",
    "Sports",
    "Religion",
    "Crime",
    "Books"
]

# ---------------------------------------------------
# 2. Convert Corex output to readable string
# ---------------------------------------------------
def topic_to_string(topic):
    return "; ".join([f"{w} ({score:.3f})" for w, score, idx in topic])

# ---------------------------------------------------
# 3. Build table for FINAL model (anchor_strength=5)
# ---------------------------------------------------
n_words = 15
topics_out = corex5.get_topics(n_words=n_words)

rows = []
for i, name in enumerate(topic_names):
    row = {
        "Topic": name,
        "Anchor words": ", ".join(anchor_words_dict[name]),
        "Top words (anchor_strength=5)": topic_to_string(topics_out[i]),
    }
    rows.append(row)

df = pd.DataFrame(rows)

# ---------------------------------------------------
# 4. Save as Excel
# ---------------------------------------------------
output_path = "corex_topics_anchor5.xlsx"
df.to_excel(output_path, index=False)

output_path


'corex_topics_anchor5.xlsx'

##### Labeling


In [46]:
import numpy as np

# 1) document–topic (n_docs, n_topics)
topic_scores = corex5.p_y_given_x  
topic_scores.shape  


(39863, 14)

In [47]:
import pandas as pd

doc_topic_df = pd.DataFrame(
    topic_scores,
    columns=topic_names 
)
doc_topic_df.head()


,Housing,Education,Union,Election,Military,Indebtedness,Family,Art,Health,Business Ownership,Sports,Religion,Crime,Books
0,0.000001,0.999999,0.000001,0.000001,0.998956,0.000001,0.999999,0.000001,0.846245,0.000001,0.000001,0.999997,0.004254,0.000335
1,0.999999,0.999999,0.995505,0.999830,0.999999,0.999999,0.999999,0.999999,0.999999,0.999999,0.999999,0.999999,0.999999,0.999999
2,0.000001,0.001451,0.000001,0.000001,0.000001,0.000001,0.000098,0.999999,0.000001,0.000073,0.999999,0.999047,0.000001,0.999999
3,0.000001,0.999905,0.000001,0.000001,0.000028,0.000001,0.041545,0.999999,0.000001,0.018908,0.999999,0.000001,0.000001,0.000001
4,0.000001,0.000001,0.999999,0.000001,0.000001,0.999999,0.000001,0.000001,0.000001,0.999999,0.000001,0.000001,0.000001,0.000001


In [48]:
scores_flat = topic_scores.flatten()

score_summary = {
    "min": float(scores_flat.min()),
    "max": float(scores_flat.max()),
    "mean": float(scores_flat.mean()),
    "median": float(np.median(scores_flat)),
    "p75": float(np.percentile(scores_flat, 75)),
    "p90": float(np.percentile(scores_flat, 90)),
    "p95": float(np.percentile(scores_flat, 95)),
    "p99": float(np.percentile(scores_flat, 99)),
}
score_summary


{'min': 1e-06,
 'max': 0.999999,
 'mean': 0.2605572711633499,
 'median': 1e-06,
 'p75': 0.9592098759537353,
 'p90': 0.999999,
 'p95': 0.999999,
 'p99': 0.999999}

In [37]:
topic_summary = pd.DataFrame({
    "topic": topic_names,
    "min":  topic_scores.min(axis=0),
    "max":  topic_scores.max(axis=0),
    "mean": topic_scores.mean(axis=0),
    "median": np.median(topic_scores, axis=0),
})
topic_summary


,topic,min,max,mean,median
0,Housing,0.000001,0.999999,0.175913,0.000001
1,Education,0.000001,0.999999,0.260442,0.000001
2,Union,0.000001,0.999999,0.258363,0.000001
3,Election,0.000001,0.999999,0.299595,0.000001
4,Military,0.000001,0.999999,0.227874,0.000001
5,Indebtedness,0.000001,0.999999,0.284666,0.000001
6,Family,0.000001,0.999999,0.268528,0.000001
7,Art,0.000001,0.999999,0.281041,0.000001
8,Health,0.000001,0.999999,0.303160,0.000001
9,Business Ownership,0.000001,0.999999,0.228446,0.000001


In [49]:
assert len(data) == len(doc_topic_df)

df_scored = pd.concat([data.reset_index(drop=True), doc_topic_df.reset_index(drop=True)], axis=1)

df_scored.shape


(39863, 29)

In [52]:
THRESH = 0.8

# Boolean matrix: (n_docs, n_topics)
label_mat = doc_topic_df.ge(THRESH)

# Add boolean columns (optional but very useful for trends)
label_cols = [f"label_{t}" for t in topic_names]
df_scored[label_cols] = label_mat.values

# Add list-of-labels per doc
df_scored["labels"] = label_mat.apply(lambda row: [t for t, v in row.items() if v], axis=1)

# Add number of labels
df_scored["n_labels"] = df_scored["labels"].apply(len)

df_scored[["title", "publisher", "date", "labels", "n_labels"]].head()


,title,publisher,date,labels,n_labels
0,A Well-Documented Childhood. A Very Private Life.,New York Times,2024-12-31,"[Education, Military, Family, Health, Religion]",5
1,These were the big stories in arts and culture...,Other publisher,2024-12-31,"[Housing, Education, Union, Election, Military...",14
2,She Exalted The Beauty Of Dance,New York Times,2024-12-31,"[Art, Sports, Religion, Books]",4
3,"Under a Highway in Rio, a Dance Style Charms a...",New York Times,2024-12-31,"[Education, Art, Sports]",3
4,"Congressional pay, minimum wage stagnant for y...",Other publisher,2024-12-31,"[Union, Indebtedness, Business Ownership]",3


In [53]:
labeling_summary = {
    "threshold": THRESH,
    "pct_with_>=1_label": (df_scored["n_labels"] >= 1).mean(),
    "avg_labels_per_doc": df_scored["n_labels"].mean(),
    "pct_with_>=3_labels": (df_scored["n_labels"] >= 3).mean(),
}
labeling_summary


{'threshold': 0.8,
 'pct_with_>=1_label': np.float64(0.9597872714045607),
 'avg_labels_per_doc': np.float64(3.5859819883099617),
 'pct_with_>=3_labels': np.float64(0.588992298622783)}

In [54]:
topic_counts = df_scored[label_cols].sum().sort_values(ascending=False)
topic_counts


label_Election              11912
label_Health                11829
label_Indebtedness          11447
label_Business Ownership    10890
label_Crime                 10569
label_Art                   10248
label_Family                10169
label_Education             10125
label_Union                 10098
label_Religion               9865
label_Books                  9529
label_Sports                 9265
label_Military               8905
label_Housing                8097
dtype: int64

#### top 3 + 0.7

In [23]:
import numpy as np
import pandas as pd

# topic_scores: (n_docs, n_topics)
topic_scores = corex5.p_y_given_x
assert topic_scores.shape[0] == len(data)
assert topic_scores.shape[1] == len(topic_names)

doc_topic_df = pd.DataFrame(topic_scores, columns=topic_names)



In [24]:
threshold = 0.7
top_k = 3

scores = topic_scores  # numpy array (n_docs, n_topics)

# Indices of topics sorted by descending score for each doc
sorted_idx = np.argsort(-scores, axis=1)  # (n_docs, n_topics)

labels_list = []
n_labels_list = []

for i in range(scores.shape[0]):
    idx_desc = sorted_idx[i]
    # filter by threshold
    idx_keep = [j for j in idx_desc if scores[i, j] >= threshold]
    # cap at top_k
    idx_keep = idx_keep[:top_k]
    labels = [topic_names[j] for j in idx_keep]
    labels_list.append(labels)
    n_labels_list.append(len(labels))

data_labeled = data.copy()
data_labeled["labels"] = labels_list
data_labeled["n_labels"] = n_labels_list

# Make per-topic binary columns for counting (one doc can count in multiple topics)
for t in topic_names:
    data_labeled[f"label_{t}"] = data_labeled["labels"].apply(lambda lst: int(t in lst))


In [21]:
summary = {
    "threshold": threshold,
    "top_k": top_k,
    "pct_with_>=1_label": float((data_labeled["n_labels"] >= 1).mean()),
    "pct_with_>=2_labels": float((data_labeled["n_labels"] >= 2).mean()),
    "pct_with_>=3_labels": float((data_labeled["n_labels"] >= 3).mean()),
    "avg_labels_per_doc": float(data_labeled["n_labels"].mean()),
    "max_labels_any_doc": int(data_labeled["n_labels"].max()),
}
summary


{'threshold': 0.7,
 'top_k': 3,
 'pct_with_>=1_label': 0.9349522113237839,
 'pct_with_>=2_labels': 0.7136191455735895,
 'pct_with_>=3_labels': 0.5060080776660061,
 'avg_labels_per_doc': 2.1545794345633795,
 'max_labels_any_doc': 3}

In [22]:
topic_counts = data_labeled[[f"label_{t}" for t in topic_names]].sum().sort_values(ascending=False)
topic_counts


label_Election              11448
label_Art                    9279
label_Education              8687
label_Indebtedness           8527
label_Union                  7813
label_Military               7646
label_Housing                7549
label_Family                 6532
label_Health                 5624
label_Business Ownership     4860
label_Sports                 4140
label_Religion               3783
dtype: int64

In [23]:
# Parse year safely (your date is object; assume "YYYY-MM-DD")
data_labeled["date"] = pd.to_datetime(data_labeled["date"], errors="coerce")
data_labeled["year"] = data_labeled["date"].dt.year

# Publisher split
data_labeled["publisher_group"] = np.where(
    data_labeled["publisher"].str.lower().str.contains("new york times", na=False),
    "NYT",
    "Other"
)

# Keep only rows with valid year
trend_df = data_labeled.dropna(subset=["year"]).copy()
trend_df["year"] = trend_df["year"].astype(int)

# Total articles per year/group
totals = (
    trend_df
    .groupby(["publisher_group", "year"], as_index=False)
    .size()
    .rename(columns={"size": "n_articles"})
)

# Topic counts per year/group (one article can contribute to multiple topics)
topic_count_long = (
    trend_df
    .melt(
        id_vars=["publisher_group", "year"],
        value_vars=[f"label_{t}" for t in topic_names],
        var_name="topic",
        value_name="is_labeled"
    )
)

# Clean topic names (remove "label_")
topic_count_long["topic"] = topic_count_long["topic"].str.replace("^label_", "", regex=True)

counts = (
    topic_count_long
    .groupby(["publisher_group", "year", "topic"], as_index=False)["is_labeled"]
    .sum()
    .rename(columns={"is_labeled": "n_labeled"})
)

# Merge totals and compute share
time_trend = counts.merge(totals, on=["publisher_group", "year"], how="left")
time_trend["share"] = time_trend["n_labeled"] / time_trend["n_articles"]

time_trend.head()


,publisher_group,year,topic,n_labeled,n_articles,share
0,NYT,1980,Art,33,184,0.179348
1,NYT,1980,Business Ownership,10,184,0.054348
2,NYT,1980,Education,29,184,0.157609
3,NYT,1980,Election,47,184,0.255435
4,NYT,1980,Family,15,184,0.081522


In [25]:
import numpy as np
import pandas as pd

# -----------------------------
# 0) Basic checks
# -----------------------------
assert topic_scores.shape[0] == len(data), "topic_scores rows must match data rows"
assert topic_scores.shape[1] == len(topic_names), "topic_scores cols must match topic_names length"

# -----------------------------
# 1) Build doc-topic score dataframe
# -----------------------------
doc_topic_df = pd.DataFrame(topic_scores, columns=topic_names)

# -----------------------------
# 2) Prepare base doc-level dataframe for sampling
#    - publisher_group: NYT vs Other
#    - year: for quick inspection (optional)
# -----------------------------
df_base = data.copy()

# If your publisher column already has "New York Times" vs "Other publisher", keep it.
# Otherwise adjust this mapping logic.
df_base["publisher_group"] = np.where(
    df_base["publisher"].str.contains("New York Times", case=False, na=False),
    "NYT",
    "Other"
)

# Parse year (date is string like '2024-12-31')
df_base["date"] = pd.to_datetime(df_base["date"], errors="coerce")
df_base["year"] = df_base["date"].dt.year

# Combine with topic scores
df_all = pd.concat([df_base.reset_index(drop=True), doc_topic_df.reset_index(drop=True)], axis=1)

# -----------------------------
# 3) (Optional) Your labeling rule: top_k + threshold
# -----------------------------
def assign_labels_topk_threshold(scores_row, topic_names, top_k=3, threshold=0.7):
    """
    scores_row: 1D array-like of length n_topics
    returns: list of topic names
    """
    scores = np.asarray(scores_row)
    top_idx = np.argsort(scores)[::-1][:top_k]
    chosen = [topic_names[i] for i in top_idx if scores[i] >= threshold]
    return chosen

def add_labels(df, topic_names, top_k=3, threshold=0.7):
    score_mat = df[topic_names].to_numpy()
    labels = [assign_labels_topk_threshold(score_mat[i], topic_names, top_k, threshold)
              for i in range(score_mat.shape[0])]
    out = df.copy()
    out["labels"] = labels
    out["n_labels"] = out["labels"].apply(len)
    return out

# If you want sampling strategy B, uncomment:
# df_all = add_labels(df_all, topic_names, top_k=3, threshold=0.7)

# -----------------------------
# 4) Clean sampling function
# -----------------------------
def sample_articles(
    df,
    topic_names,
    n_per_group=5,
    mode="top_score",     # "top_score" or "label_rule"
    top_k=3,
    threshold=0.7,
    random_tiebreak=True,
    seed=42
):
    """
    mode="top_score": for each (topic, publisher_group), pick top n_per_group by topic score.
    mode="label_rule": first compute labels via (top_k+threshold), then sample only docs where topic in labels,
                       still ranked by topic score.

    Returns a tidy long dataframe with samples.
    """
    rng = np.random.default_rng(seed)

    work = df.copy()

    # Ensure publisher_group exists
    if "publisher_group" not in work.columns:
        raise ValueError("df must have 'publisher_group' column (NYT vs Other).")

    # If mode=label_rule, create labels if not already present
    if mode == "label_rule":
        if "labels" not in work.columns:
            work = add_labels(work, topic_names, top_k=top_k, threshold=threshold)

    samples = []

    for topic in topic_names:
        for pg in ["NYT", "Other"]:
            sub = work[work["publisher_group"] == pg].copy()

            if mode == "label_rule":
                sub = sub[sub["labels"].apply(lambda x: topic in x)]

            # Nothing to sample
            if sub.empty:
                continue

            # Optional random tiebreak to avoid always same ordering when many equal scores
            if random_tiebreak:
                sub["_rand"] = rng.random(len(sub))
                sub = sub.sort_values(by=[topic, "_rand"], ascending=[False, True])
            else:
                sub = sub.sort_values(by=topic, ascending=False)

            pick = sub.head(n_per_group).copy()
            pick["sample_topic"] = topic
            pick["sample_publisher_group"] = pg
            pick["sample_topic_score"] = pick[topic].astype(float)

            # Keep only useful columns for review (edit as needed)
            keep_cols = [
                "sample_topic", "sample_publisher_group", "sample_topic_score",
                "title", "publisher", "date", "year", "section",
                "source_file"
            ]

            # Add labels columns if present
            if "labels" in pick.columns:
                keep_cols += ["labels", "n_labels"]

            # Add a short snippet for quick inspection (optional)
            if "body" in pick.columns:
                keep_cols += ["body"]

            pick = pick[[c for c in keep_cols if c in pick.columns]]
            samples.append(pick)

    if not samples:
        return pd.DataFrame()

    result = pd.concat(samples, ignore_index=True)

    # Sort for readability
    result = result.sort_values(
        by=["sample_topic", "sample_publisher_group", "sample_topic_score"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    return result

# -----------------------------
# 5) Run sampling
# -----------------------------

# A) Top-score sampling (recommended first pass)
sample_top = sample_articles(
    df_all,
    topic_names=topic_names,
    n_per_group=5,
    mode="top_score",
    seed=42
)

# B) Label-rule sampling (recommended second pass, matches your production rule)
# sample_label = sample_articles(
#     df_all,
#     topic_names=topic_names,
#     n_per_group=5,
#     mode="label_rule",
#     top_k=3,
#     threshold=0.7,
#     seed=42
# )

# Preview
sample_top.head(10)


,sample_topic,sample_publisher_group,sample_topic_score,title,publisher,date,year,section,source_file,body
0,Art,NYT,0.999999,A French Designer Who Celebrates Mexico's Popu...,New York Times,2021-08-19,2021,T-MAGAZINE,NYT/10.DOCX,"In his colorful Guadalajara work space, Fabien..."
1,Art,NYT,0.999999,No Headline In Original,New York Times,2005-03-24,2005,Section E; Column 5; The Arts/Cultural Desk; P...,NYT/56.361.DOCX,"Prize for Ha Jin\n Ha Jin, below, has won the ..."
2,Art,NYT,0.999999,Transcript: Ezra Klein Discusses Kamala Harris...,New York Times,2024-08-06,2024,OPINION,NYT/3.DOCX,"Every Tuesday and Friday, Ezra Klein invites y..."
3,Art,NYT,0.999999,No Headline In Original,New York Times,2006-06-17,2006,Section B; Column 6; The Arts/Cultural Desk; P...,NYT/56.361.DOCX,10 P.M. (Sundance) PRIDE DOUBLE FEATURE -- The...
4,Art,NYT,0.999999,The Manhattan 'Madam' Who Hobnobbed With the C...,New York Times,2021-11-02,2021,BOOKS; review,NYT/9.DOCX,"MADAM\nThe Biography of Polly Adler, Icon of t..."
5,Art,Other,0.999999,screen this,Other publisher,2019-03-29,2019,DO THIS; Pg. W16,Other publishers/Files (500) (7).DOCX,Reviews Ratings: iiii Excellent iii Good ii Fa...
6,Art,Other,0.999999,BLACK-AND-WHITE ENIGMAS AN EXHIBIT OF AMERICAN...,Other publisher,1999-07-26,1999,FEATURES MAGAZINE: ENTERTAINMENT; Pg. C05,Other publishers/Files (500) (13).DOCX,The American Dream deferred meets Socialist Re...
7,Art,Other,0.999999,BEST POP CONCERT: ELVIS COSTELLO,Other publisher,2005-12-29,2005,ARTS & ENTERTAINMENT; BEST OF 2005; Pg. W-19,Other publishers/Files (500) (23).DOCX,"If you're into rock at all, the week of July 2..."
8,Art,Other,0.999999,The Arctic Monkeys are cool with the past; Ray...,Other publisher,2006-02-21,2006,LIFE; Pg. 7B,Other publishers/Files (500) (22).DOCX,"Pop/rock: Arctic Monkeys,\nWhatever People Say..."
9,Art,Other,0.999999,* Coriolis Theater Company's Chicago area prem...,Other publisher,2013-09-06,2013,TIMEOUT; Pg. 7,Other publishers/Files (500) (21).DOCX,* Coriolis Theater Company's Chicago area prem...


In [27]:
row = sample_top.iloc[1] # change numbers to check top 10 scores of each 12 topic, 0 - 119

print("TITLE:", row["title"])
print("TOPIC:", row["sample_topic"])
print("SCORE:", row["sample_topic_score"])
print("PUBLISHER:", row["publisher"])
print("\nBODY:\n")
print(row["body"])


TITLE: No Headline In Original
TOPIC: Art
SCORE: 0.999999
PUBLISHER: New York Times

BODY:

Prize for Ha Jin
 Ha Jin, below, has won the PEN/Faulkner Award for Fiction for the second time, for his novel ''War Trash'' , about a Chinese prisoner of war held by Americans during the Korean War. Mr. Jin, who won the prize in 2000 for his novel ''Waiting,'' will receive $15,000 at the 25th annual ceremony at the Folger Shakespeare Library in Washington on May 14. The other four finalists are each to receive $5,000. They are Jerome Charyn for ''The Green Lantern'' (Thunder's Mouth Press), Edwidge Danticat for ''The Dew Breaker'' (Alfred A. Knopf), Marilynne Robinson for ''Gilead'' (Farrar, Straus & Giroux) and Steve Yarbrough for ''Prisoners of War'' (Alfred A. Knopf).

Pop Charts: 50 Cent Again
 50 Cent's ''Massacre'' (Interscope) tops the Billboard album charts again this week, but just barely. The album sold 364,000 copies last week, according to Nielsen SoundScan, a drop of more than 50 p